# 1. Setup & Environment
- 대용량 데이터 처리용 라이브러리 로드
- 부동소수점 포맷 및 디스플레이 환경 설정

In [1]:
# 1. 라이브러리 로드 및 환경 설정
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda x: "%.4f" % x)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

# 2. Configuration & Inequality Scenario Definitions
- 생활인구 및 접근성 산출물 경로 설정
- 6개 시간대·교통 조건별 분석 시나리오(콤보) 정의

In [2]:
# 2. 경로 및 분석 시나리오(콤보) 정의
BASE_DIR = Path("/mnt/cowork/EV")
D1_FP = BASE_DIR / "input/processed/서울시_생활인구/집계구_생활인구_원본(OA-14979)/d1_final_2021_2024.csv"

# 접근성 산출물 기본 경로 (Gaussian)
DIR_GAUSSIAN = BASE_DIR / "output/g2sfca_sfast_final_gaussian"
DIR_SIMYA_GAUSSIAN = BASE_DIR / "output/g2sfca_sfast_simya_gaussian"
DIR_OUTPUT = BASE_DIR / "output"
DIR_OUTPUT.mkdir(parents=True, exist_ok=True)

YEARS = [2021, 2022, 2023, 2024]

# 불평등 지수 산출 대상 시나리오 정의: (combo_label, file_tag_pattern, pop_col)
COMBOS = [
    ("week_오전_congested", "{year}_week_오전_congested", "오전_avg"),
    ("week_낮_normal", "{year}_week_낮_normal", "낮_avg"),
    ("weekend_오전_freeflow", "{year}_weekend_오전_freeflow", "오전_avg"),
    ("weekend_낮_normal", "{year}_weekend_낮_normal", "낮_avg"),
    ("week_심야_freeflow", "{year}_week_심야_freeflow", "심야_avg"),
    ("weekend_심야_freeflow", "{year}_weekend_심야_freeflow", "심야_avg")
]

# 3. Inequality Metrics Functions (Gini & Palma)
- 인구가중 로렌츠 곡선 기반 Gini 계수 산출 함수
- 경계구간 선형보간(Linear Interpolation) 적용 Palma 비율 산출 함수

In [3]:
# 3. 인구가중 로렌츠 곡선 기반 지니계수 및 팔마비율 산출 함수
def weighted_gini(score: np.ndarray, weight: np.ndarray) -> float:
    """인구가중 Gini 계수 (Trapezoidal Integration).
    - score: 집계구 1인당 접근성 점수
    - weight: 집계구 생활인구 수요
    """
    df = pd.DataFrame({"score": score, "weight": weight})
    df = df[df["weight"] > 0].dropna()
    if len(df) == 0 or df["weight"].sum() <= 0:
        return np.nan
    
    df = df.sort_values("score").reset_index(drop=True)
    df["mass"] = df["score"] * df["weight"]  # 총 접근가능 공급량
    total_w = df["weight"].sum()
    total_mass = df["mass"].sum()
    if total_mass <= 0:
        return np.nan
    
    cum_w = np.concatenate([[0.0], (df["weight"].cumsum() / total_w).values])
    cum_mass = np.concatenate([[0.0], (df["mass"].cumsum() / total_mass).values])
    
    # Lorenz curve 곡선하면적 적분을 통한 지니계수 계산
    gini = 1.0 - np.sum((cum_w[1:] - cum_w[:-1]) * (cum_mass[1:] + cum_mass[:-1]))
    return float(gini)


def weighted_palma(score: np.ndarray, weight: np.ndarray) -> float:
    """인구가중 Palma 비율 (상위 10% 점유율 / 하위 40% 점유율).
    - 경계선 집계구는 인구 비중에 맞춰 선형보간(Linear Interpolation) 처리
    """
    df = pd.DataFrame({"score": score, "weight": weight})
    df = df[df["weight"] > 0].dropna()
    if len(df) == 0:
        return np.nan
    
    df = df.sort_values("score").reset_index(drop=True)
    df["mass"] = df["score"] * df["weight"]
    total_w = df["weight"].sum()
    total_mass = df["mass"].sum()
    if total_mass <= 0 or total_w <= 0:
        return np.nan

    cum_w_end = df["weight"].cumsum()
    cum_w_start = cum_w_end - df["weight"]
    w_share_start = (cum_w_start / total_w).values
    w_share_end = (cum_w_end / total_w).values
    mass_vals = df["mass"].values

    def mass_in_range(lo: float, hi: float) -> float:
        overlap_lo = np.clip(w_share_start, lo, 1.0)
        overlap_hi = np.clip(w_share_end, 0.0, hi)
        overlap = np.clip(overlap_hi - overlap_lo, 0.0, None)
        span = w_share_end - w_share_start
        frac = np.where(span > 0, overlap / span, 0.0)
        return float(np.sum(frac * mass_vals))

    bottom40_mass = mass_in_range(0.0, 0.40)
    top10_mass = mass_in_range(0.90, 1.00)

    if bottom40_mass <= 0:
        return np.nan
    return float(top10_mass / bottom40_mass)

# 4. Data Loader & Path Resolver
- 연도별 생활인구 수요 데이터 로더
- 표준 산출물(`_mw.csv`) 및 기존 산출물 경로 자동 탐색 매퍼

In [4]:
# 4. 생활인구 원본 데이터 로드 및 딕셔너리 빌더
df_pop_all = pd.read_csv(D1_FP, dtype={"집계구코드": str})
print(f">> 생활인구 데이터 로드 완료: 총 {len(df_pop_all):,}개 행")

def get_pop_series(year: int, pop_col: str) -> pd.Series:
    """연도 및 시간대별 인구수 Series 반환"""
    df_y = df_pop_all[df_pop_all["year"] == year]
    return pd.Series(df_y[pop_col].astype(float).values, index=df_y["집계구코드"].values)

def resolve_score_file(year: int, tag_tmpl: str) -> Path:
    """_mw 파일 우선 탐색 후 원본 파일 경로 반환"""
    tag = tag_tmpl.format(year=year)
    
    # 1. 신규 표준 산출물 (_mw.csv)
    fp_new = DIR_GAUSSIAN / f"g2sfca_score_{tag}_mw.csv"
    if fp_new.exists():
        return fp_new
    
    # 2. 통합 폴더 내 기본 csv
    fp_orig = DIR_GAUSSIAN / f"g2sfca_score_{tag}.csv"
    if fp_orig.exists():
        return fp_orig
    
    # 3. 심야 전용 구버전 파일 매핑
    if "심야" in tag:
        daytype = "weekend" if "weekend" in tag else "week"
        fp_simya = DIR_SIMYA_GAUSSIAN / f"g2sfca_score_{year}_{daytype}_심야.csv"
        if fp_simya.exists():
            return fp_simya
            
    return None

>> 생활인구 데이터 로드 완료: 총 76,612개 행


# 5. Batch Inequality Computation Engine
- 연도별·시나리오별 Gini 계수 및 Palma 비율 일괄 산출
- 집계구별 인구 가중치 매핑 및 결측 검증

In [5]:
# 5. 연도 및 시나리오별 Gini / Palma 일괄 계산 파이프라인
print("=" * 80)
print("RUNNING: POPULATION-WEIGHTED GINI & PALMA INEQUALITY PIPELINE")
print("=" * 80)

records = []

for year in YEARS:
    for combo_label, tag_tmpl, pop_col in COMBOS:
        fp_score = resolve_score_file(year, tag_tmpl)
        
        if fp_score is None:
            print(f"  [!] 파일 누락 스킵: {year} | {combo_label}")
            continue
            
        # 데이터 로드
        df_score = pd.read_csv(fp_score, dtype={"oa_code": str})
        s_pop = get_pop_series(year, pop_col)
        
        # 인구 데이터 매핑
        df_score["pop"] = df_score["oa_code"].map(s_pop)
        n_missing_pop = df_score["pop"].isna().sum()
        df_valid = df_score.dropna(subset=["pop"])
        
        # 불평등 지수 산출
        score_arr = df_valid["accessibility_score"].values
        pop_arr = df_valid["pop"].values
        
        val_gini = weighted_gini(score_arr, pop_arr)
        val_palma = weighted_palma(score_arr, pop_arr)
        total_pop = pop_arr.sum()
        
        print(f"  [>] {year} | {combo_label:<25} | Gini: {val_gini:.4f} | Palma: {val_palma:6.2f} | Pop: {total_pop:,.0f}")
        
        records.append({
            "year": year,
            "combo": combo_label,
            "decay": "gaussian",
            "gini": val_gini,
            "palma": val_palma,
            "n_oa": len(df_valid),
            "n_missing_pop": n_missing_pop,
            "total_pop": total_pop
        })

df_inequality = pd.DataFrame(records)

RUNNING: POPULATION-WEIGHTED GINI & PALMA INEQUALITY PIPELINE
  [>] 2021 | week_오전_congested         | Gini: 0.2743 | Palma:   0.90 | Pop: 10,424,451
  [>] 2021 | week_낮_normal             | Gini: 0.2590 | Palma:   0.85 | Pop: 10,683,729
  [>] 2021 | weekend_오전_freeflow       | Gini: 0.1193 | Palma:   0.42 | Pop: 10,424,451
  [>] 2021 | weekend_낮_normal          | Gini: 0.2709 | Palma:   0.91 | Pop: 10,683,729
  [>] 2021 | week_심야_freeflow          | Gini: 0.1414 | Palma:   0.46 | Pop: 10,146,230
  [>] 2021 | weekend_심야_freeflow       | Gini: 0.1401 | Palma:   0.45 | Pop: 10,146,230
  [>] 2022 | week_오전_congested         | Gini: 0.2438 | Palma:   0.78 | Pop: 10,427,556
  [>] 2022 | week_낮_normal             | Gini: 0.2303 | Palma:   0.76 | Pop: 10,724,828
  [>] 2022 | weekend_오전_freeflow       | Gini: 0.1120 | Palma:   0.41 | Pop: 10,427,556
  [>] 2022 | weekend_낮_normal          | Gini: 0.2293 | Palma:   0.76 | Pop: 10,724,828
  [>] 2022 | week_심야_freeflow          | Gini: 0.1409 | Pa

# 6. Summary Pivot Tables & Longitudinal Trends
- 2021~2024 시나리오별 불평등 지수 시계열 추이 분석
- 다중 인덱스 피벗 테이블 및 기간 내 변화율(%) 출력

In [6]:
# 6. 연도별(2021->2024) 불평등 추이 MultiIndex 피벗 분석
print("\n" + "=" * 80)
print("                       시나리오별 지니계수(Gini) 추이 (2021~2024)")
print("=" * 80)
piv_gini = df_inequality.pivot(index="combo", columns="year", values="gini")
piv_gini["변화율(%)"] = ((piv_gini[2024] - piv_gini[2021]) / piv_gini[2021]) * 100
display(piv_gini)

print("\n" + "=" * 80)
print("                      시나리오별 팔마비율(Palma) 추이 (2021~2024)")
print("=" * 80)
piv_palma = df_inequality.pivot(index="combo", columns="year", values="palma")
piv_palma["변화율(%)"] = ((piv_palma[2024] - piv_palma[2021]) / piv_palma[2021]) * 100
display(piv_palma)


                       시나리오별 지니계수(Gini) 추이 (2021~2024)


year,2021,2022,2023,2024,변화율(%)
combo,,,,,
week_낮_normal,0.2590,0.2303,0.2895,0.2693,3.9893
week_심야_freeflow,0.1414,0.1409,0.1256,0.1242,-12.1570
week_오전_congested,0.2743,0.2438,0.2838,0.2613,-4.7564
weekend_낮_normal,0.2709,0.2293,0.2860,0.2708,-0.0461
weekend_심야_freeflow,0.1401,0.1379,0.1272,0.1269,-9.4259
weekend_오전_freeflow,0.1193,0.1120,0.1439,0.1548,29.7489



                      시나리오별 팔마비율(Palma) 추이 (2021~2024)


year,2021,2022,2023,2024,변화율(%)
combo,,,,,
week_낮_normal,0.8519,0.7582,1.0108,0.9153,7.4450
week_심야_freeflow,0.4584,0.4666,0.4377,0.4330,-5.5387
week_오전_congested,0.9022,0.7769,0.9767,0.8845,-1.9667
weekend_낮_normal,0.9057,0.7575,1.0010,0.9268,2.3246
weekend_심야_freeflow,0.4515,0.4637,0.4373,0.4390,-2.7896
weekend_오전_freeflow,0.4221,0.4096,0.4759,0.5153,22.0753


# 7. Sanity Check & Output Export
- 지수 범위 및 인구 결측 무결성 검증
- 최종 불평등 분석 결과 CSV(`_mw.csv`) 내보내기

In [7]:
# 7. 데이터 무결성 검증 및 최종 CSV 내보내기
print("\n=== [Sanity Check] ===")
print(f"- Gini 계수 범위  : {df_inequality['gini'].min():.4f} ~ {df_inequality['gini'].max():.4f}")
print(f"- Palma 비율 범위 : {df_inequality['palma'].min():.4f} ~ {df_inequality['palma'].max():.4f}")
print(f"- 인구 결측 집계구: 총 {df_inequality['n_missing_pop'].sum()}건")

# 최종 결과 저장 (_mw)
out_fp = DIR_OUTPUT / "gini_palma_2021_2024_mw.csv"
export_cols = ["year", "combo", "decay", "gini", "palma", "n_oa", "total_pop"]
df_inequality[export_cols].to_csv(out_fp, index=False, encoding="utf-8-sig")

print(f"\n>> 분석 결과 저장 완료: {out_fp}")


=== [Sanity Check] ===
- Gini 계수 범위  : 0.1120 ~ 0.2895
- Palma 비율 범위 : 0.4096 ~ 1.0108
- 인구 결측 집계구: 총 0건

>> 분석 결과 저장 완료: /mnt/cowork/EV/output/gini_palma_2021_2024_mw.csv


# 8. Result Validation (Comparison with Original Outputs)
- 원본 Gini/Palma 산출물(`gini_palma_2021_2024.csv`)과 신규 산출물(`_mw.csv`) 간 수치 오차 검증
- Gaussian 감쇄 모델 기준 시나리오별 지니계수 및 팔마비율 정밀도 비교 요약표 출력

In [9]:
# 8. 원본 Gini / Palma 산출물 vs 신규 산출물(_mw) 정밀 오차 검증
fp_orig_summary = DIR_OUTPUT / "gini_palma_2021_2024.csv"
fp_new_summary = DIR_OUTPUT / "gini_palma_2021_2024_mw.csv"

if not fp_orig_summary.exists():
    print(f"[!] 비교할 원본 결과 파일이 존재하지 않습니다: {fp_orig_summary}")
elif not fp_new_summary.exists():
    print(f"[!] 신규 산출물 파일이 생성되지 않았습니다: {fp_new_summary}")
else:
    # 1. 데이터 로드
    df_orig = pd.read_csv(fp_orig_summary)
    df_new = pd.read_csv(fp_new_summary)
    
    # 2. Gaussian 감쇄 모델 기준 필터링
    df_orig_g = df_orig[df_orig["decay"] == "gaussian"].copy()
    df_new_g = df_new[df_new["decay"] == "gaussian"].copy()
    
    # 원본 파일에서 심야 라벨이 '심야'로만 되어 있을 경우 매핑 보정
    df_orig_g["combo_clean"] = df_orig_g["combo"].replace({
        "심야": "week_심야_freeflow"
    })
    
    # 3. 데이터 병합 (Year + Scenario Combo 기준)
    comp = df_orig_g.merge(
        df_new_g, 
        left_on=["year", "combo_clean"], 
        right_on=["year", "combo"], 
        suffixes=("_orig", "_new"), 
        how="inner"
    )
    
    if len(comp) == 0:
        print("[!] 일치하는 시나리오 항목을 찾을 수 없습니다. combo 명칭을 확인해주세요.")
    else:
        # 4. 절대 오차(Absolute Difference) 계산
        comp["gini_diff"] = (comp["gini_orig"] - comp["gini_new"]).abs()
        comp["palma_diff"] = (comp["palma_orig"] - comp["palma_new"]).abs()
        
        # 허용 오차 기준 판정 (Gini: 1e-5 미만, Palma: 1e-4 미만)
        comp["Status"] = np.where(
            (comp["gini_diff"] < 1e-5) & (comp["palma_diff"] < 1e-4),
            "완전 일치 (정상)",
            "오차 발생 확인 필요"
        )
        
        # 출력용 시나리오 컬럼 명시적 지정
        comp["Scenario"] = comp["combo_clean"]
        
        # 5. 요약 테이블 출력
        display_cols = [
            "year", "Scenario", "Status",
            "gini_orig", "gini_new", "gini_diff",
            "palma_orig", "palma_new", "palma_diff"
        ]
        
        print("=" * 95)
        print("           인구가중 Gini 계수 및 Palma 비율 원본 vs 리팩토링 코드 수치 검증 요약표")
        print("=" * 95)
        display(comp[display_cols])
        
        # 최대/평균 오차 요약 출력
        print("\n=== [정밀 오차 요약] ===")
        print(f"- Gini 계수 최대 오차 : {comp['gini_diff'].max():.10e} (평균: {comp['gini_diff'].mean():.10e})")
        print(f"- Palma 비율 최대 오차: {comp['palma_diff'].max():.10e} (평균: {comp['palma_diff'].mean():.10e})")
        
        if comp["gini_diff"].max() < 1e-5 and comp["palma_diff"].max() < 1e-4:
            print(">> [판정] 모든 시나리오의 불평등 지수가 원본 산출물과 수학적으로 완벽히 일치합니다.")

           인구가중 Gini 계수 및 Palma 비율 원본 vs 리팩토링 코드 수치 검증 요약표


,year,Scenario,Status,gini_orig,gini_new,gini_diff,palma_orig,palma_new,palma_diff
0,2021,week_오전_congested,완전 일치 (정상),0.2743,0.2743,0.0000,0.9022,0.9022,0.0000
1,2021,week_낮_normal,완전 일치 (정상),0.2590,0.2590,0.0000,0.8519,0.8519,0.0000
2,2021,weekend_오전_freeflow,완전 일치 (정상),0.1193,0.1193,0.0000,0.4221,0.4221,0.0000
3,2021,weekend_낮_normal,완전 일치 (정상),0.2709,0.2709,0.0000,0.9057,0.9057,0.0000
4,2021,week_심야_freeflow,오차 발생 확인 필요,0.1422,0.1414,0.0008,0.4602,0.4584,0.0018
5,2022,week_오전_congested,완전 일치 (정상),0.2438,0.2438,0.0000,0.7769,0.7769,0.0000
6,2022,week_낮_normal,완전 일치 (정상),0.2303,0.2303,0.0000,0.7582,0.7582,0.0000
7,2022,weekend_오전_freeflow,완전 일치 (정상),0.1120,0.1120,0.0000,0.4096,0.4096,0.0000
8,2022,weekend_낮_normal,완전 일치 (정상),0.2293,0.2293,0.0000,0.7575,0.7575,0.0000
9,2022,week_심야_freeflow,오차 발생 확인 필요,0.1406,0.1409,0.0003,0.4654,0.4666,0.0012



=== [정밀 오차 요약] ===
- Gini 계수 최대 오차 : 8.4122801092e-04 (평균: 7.8475830839e-05)
- Palma 비율 최대 오차: 1.7836299007e-03 (평균: 1.7210836764e-04)
